In [ ]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
from pathlib import Path
import httpx
import pymupdf
from src.researchos.domain.models import Paper

# PAPERS_DIR = Path(__file__).parent.parent.parent.parent / "data" / "papers"
PAPERS_DIR = Path("../data/papers")

def extract_text_pdf(paper: Paper) -> str:
    url = paper.pdf_url
    pdf_name = paper.authors[0].lower().strip()
    pdf_name = re.sub(r'[^a-z0-9_]', '_', pdf_name)
    pdf_name = pdf_name + '_' + paper.published_date.strftime('%Y')
    local_pdf_path = PAPERS_DIR / f"{pdf_name}.pdf"

    PAPERS_DIR.mkdir(parents=True, exist_ok=True)

    response = httpx.get(url)
    response.raise_for_status()

    # save pdf in local as .pdf
    with open(local_pdf_path, 'wb') as f:
        
        f.write(response.content)

    # extract text
    full_text = ""
    doc = pymupdf.open(local_pdf_path)
    for page in doc:
        full_text += page.get_text()
    
    if not full_text.strip():
        raise ValueError(f"PDF has no extractable text: {url}")

    return full_text 

        

In [ ]:
from src.researchos.infrastructure.data.arxiv import search_papers

q= "LLM-agents"
m = 4
results = await search_papers(query=q, max_results=m)

In [ ]:
results

In [ ]:
[extract_text_pdf(paper=results[i]) for i in range(4)]

# Probar desde módulo

In [ ]:
from src.researchos.infrastructure.data.arxiv import search_papers

q= "chaotic"
m = 1
results = await search_papers(query=q, max_results=m)

In [ ]:
from src.researchos.application.services.ingestion_service import extract_text_pdf

extract_text_pdf(paper=results[0])

In [ ]:
results[0].authors